# Lab - The Agentic Loop in Code

Three tasks that build the loop that lets Claude use several tools in a row until the job
is done. Real requests take more than one step, and the exam expects you to read this loop.

| Task | What you learn |
| --- | --- |
| 1 | A dictionary maps each tool name to its function, so code can run any tool Claude asks for |
| 2 | Every tool_use block gets its own tool_result, with the id of that block |
| 3 | The loop: send, read stop_reason, run the tools, repeat until end_turn |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery. "
    "Use the tools to look up orders and to start returns. "
    "Answer in one or two sentences."
)

ORDERS = {
    "ORD-12345678": {
        "item": "jacket",
        "status": "delivered",
        "delivered_days_ago": 20,
    },
}

# Two tools: look the order up, then start the return.
TOOLS = [
    {
        "name": "lookup_order",
        "description": "Look up an order by its ID. Returns the "
                       "item, the status and how many days ago "
                       "it was delivered.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "start_return",
        "description": "Start a return for a delivered order that "
                       "is inside the 30-day window. Returns the "
                       "return label number.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
]


def lookup_order(order_id):
    return ORDERS.get(order_id, {"error": "order not found"})


def start_return(order_id):
    return {"return_label": "RET-2048", "order_id": order_id}

---

## Task 1 - Map each tool name to its function

In [ ]:
# ============================================================
# TASK 1 - Map each tool name to its function
# ============================================================
#
# WHAT TO DO
#   Fill in TOOL_FUNCTIONS: each tool's name maps to the
#   function behind it. Then run a tool the way the loop will:
#   by the name and the input a tool_use block carries.
#
# WHY IT MATTERS
#   Claude can ask for either tool, in any order. Your code
#   cannot know in advance which one comes next - so it looks
#   the function up by the name in the block and calls it with
#   the input Claude chose. One dictionary, and the loop can run
#   whichever tool Claude asks for.
#
# WHERE TO SEE IT IN THE LECTURE
#   "a small dictionary maps each tool name to its function".
#
# HOW TO DO IT
#   Three blanks:
#
#       lookup_order   the function itself - no quotes, no ()
#       start_return   the same for the second tool
#       name           the tool name the block carries
# ============================================================

# BEGIN SOLUTION
TOOL_FUNCTIONS = {"lookup_order": lookup_order,
                  "start_return": start_return}

name, args = "lookup_order", {"order_id": "ORD-12345678"}   # what a tool_use block carries
try:
    output = TOOL_FUNCTIONS[name](**args)
# SCAFFOLD: TOOL_FUNCTIONS = {"lookup_order": ...,    # lookup_order - the function itself
# SCAFFOLD:                   "start_return": ...}    # start_return
# SCAFFOLD:
# SCAFFOLD: name, args = "lookup_order", {"order_id": "ORD-12345678"}   # what a tool_use block carries
# SCAFFOLD: try:
# SCAFFOLD:     output = TOOL_FUNCTIONS[...](**args)   # name - the tool Claude asked for
# END SOLUTION: replace each ... below with the value named beside it
except (TypeError, KeyError, AttributeError) as exc:   # provided: a blank left above
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(name, "->", output)

check("tool_map", tool_functions=TOOL_FUNCTIONS, output=output)

---

## Task 2 - A tool_result for every tool_use block

In [ ]:
# ============================================================
# TASK 2 - A tool_result for every tool_use block
# ============================================================
#
# WHAT TO DO
#   Finish run_tools(): for every tool_use block in Claude's
#   reply, run the matching function and collect a tool_result
#   with the id of that block. Return the whole list.
#
# WHY IT MATTERS
#   One reply can hold more than one tool_use block, and every
#   one of them needs its own result - the API rejects a
#   conversation where a tool_use goes unanswered. The id ties
#   each result to the block that asked for it. All the results
#   go back to Claude together, in one user message.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Then, for every tool use block in it".
#
# HOW TO DO IT
#   Three blanks:
#
#       block.name    picks the function from TOOL_FUNCTIONS
#       block.input   the arguments Claude chose
#       block.id      ties the result to its block
# ============================================================

# BEGIN SOLUTION
def run_tools(response):
    results = []
    for block in response.content:
        if block.type == "tool_use":
            print(f"  run {block.name}({block.input})")
            output = TOOL_FUNCTIONS[block.name](**block.input)
            results.append({"type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(output)})
    return results
# SCAFFOLD: def run_tools(response):
# SCAFFOLD:     results = []
# SCAFFOLD:     for block in response.content:
# SCAFFOLD:         if block.type == "tool_use":
# SCAFFOLD:             print(f"  run {block.name}({block.input})")
# SCAFFOLD:             output = TOOL_FUNCTIONS[...](**...)   # block.name, block.input
# SCAFFOLD:             results.append({"type": "tool_result",
# SCAFFOLD:                             "tool_use_id": ...,   # block.id
# SCAFFOLD:                             "content": json.dumps(output)})
# SCAFFOLD:     return results
# END SOLUTION: replace each ... below with the value named beside it

# provided: one real reply from Claude, to try run_tools on
question = [{"role": "user", "content":
             "I want to return the jacket from order ORD-12345678."}]
first = client.messages.create(model=MODEL, max_tokens=500,
                               output_config={"effort": "low"},
                               system=SYSTEM, tools=TOOLS, messages=question)
try:
    results = run_tools(first)
except (TypeError, KeyError, AttributeError) as exc:   # provided: a blank left above
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(results)

check("every_result", run_tools=run_tools, results=results)

---

## Task 3 - Loop until Claude stops asking for tools

In [ ]:
# ============================================================
# TASK 3 - Loop until Claude stops asking for tools
# ============================================================
#
# WHAT TO DO
#   Write the loop: send the whole conversation with the tools,
#   print the stop_reason, and leave when Claude is done.
#   Otherwise add Claude's reply and the results of its tools
#   to the conversation, and go round again.
#
# WHY IT MATTERS
#   This request takes two tools in order, with a decision in
#   between: look the order up, check the delivery date, only
#   then start the return. Claude decides which tool to call
#   and in what order. Your loop runs them - and your loop
#   decides when to stop, by reading the stop_reason. If it
#   forgot to send the results back, Claude would never see
#   them.
#
# WHERE TO SEE IT IN THE LECTURE
#   "It runs until Claude stops asking for tools".
#
# HOW TO DO IT
#   Three blanks:
#
#       "tool_use"         anything else means Claude is done
#       response.content   Claude's reply, exactly as it came
#       response           run_tools() runs every block in it
# ============================================================

messages = [{"role": "user", "content":
             "I want to return the jacket from order ORD-12345678."}]

turn = 0
try:
    # BEGIN SOLUTION
    while True:
        turn += 1
        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            output_config={"effort": "low"},
            system=SYSTEM,
            tools=TOOLS,
            messages=messages,
        )
        print(f"turn {turn}: stop_reason = {response.stop_reason}")
        if response.stop_reason != "tool_use":
            break
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": run_tools(response)})
        if turn == 10:   # provided: a safety net for this lab
            raise RuntimeError("Ten turns and Claude is still asking for tools. "
                               "Check what goes back into messages each turn.")
    # SCAFFOLD: while True:
    # SCAFFOLD:     turn += 1
    # SCAFFOLD:     response = client.messages.create(
    # SCAFFOLD:         model=MODEL,
    # SCAFFOLD:         max_tokens=500,
    # SCAFFOLD:         output_config={"effort": "low"},
    # SCAFFOLD:         system=SYSTEM,
    # SCAFFOLD:         tools=TOOLS,
    # SCAFFOLD:         messages=messages,
    # SCAFFOLD:     )
    # SCAFFOLD:     print(f"turn {turn}: stop_reason = {response.stop_reason}")
    # SCAFFOLD:     if response.stop_reason != ...:   # "tool_use" - anything else: Claude is done
    # SCAFFOLD:         break
    # SCAFFOLD:     messages.append({"role": "assistant", "content": ...})   # response.content
    # SCAFFOLD:     messages.append({"role": "user", "content": run_tools(...)})   # response
    # SCAFFOLD:     if turn == 10:   # provided: a safety net for this lab
    # SCAFFOLD:         raise RuntimeError("Ten turns and Claude is still asking for tools. "
    # SCAFFOLD:                            "Check what goes back into messages each turn.")
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError, KeyError, AttributeError) as exc:   # provided: a blank left above
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

for block in response.content:
    if block.type == "text":
        print(block.text)

check("loop", response=response, messages=messages, turn=turn)

---

## Done

Send the conversation with the tools, read the stop_reason, run every `tool_use` block,
return every result with its id, and repeat until the stop_reason is `end_turn`. Claude
decided which tools to call and in what order; your loop ran them and decided when to
stop.